# LangChain Hands-on 01 — LLMs & Chat Models
**Trainer:** Ajeetkumar

**What we will practise**
- Call an LLM through LangChain (Groq, Gemini, OpenAI, Ollama)
- invoke, stream, batch, async
- Messages: System, Human, AI
- Model parameters: temperature, max_tokens
- Response object, token usage, model profile
- One function for any provider: `init_chat_model`

> Run on Google Colab. Add your Groq key in Colab **Secrets** (🔑 icon) with the name `groq_key` (free key: console.groq.com).

# What is a Chat Model?
- A chat model takes **messages** as input and returns a **message** as output.
- LangChain gives one common interface for every provider → change the provider, keep the same code.
- Every model has the same methods: `invoke()`, `stream()`, `batch()`, `ainvoke()`.

| Provider | LangChain package | Class |
|---|---|---|
| Groq (free, fast) | `langchain-groq` | `ChatGroq` |
| Google Gemini | `langchain-google-genai` | `ChatGoogleGenerativeAI` |
| OpenAI | `langchain-openai` | `ChatOpenAI` |
| Anthropic | `langchain-anthropic` | `ChatAnthropic` |
| Local (Ollama) | `langchain-ollama` | `ChatOllama` |

In [ ]:
! pip install -q langchain langchain-groq

In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_key')

In [ ]:
import os
os.environ['GROQ_API_KEY'] = groq_api_key

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

> **Note:** Groq retired `llama-3.1-8b-instant` and `llama-3.3-70b-versatile` for free/dev users on 16 Aug 2026.
> Use `openai/gpt-oss-20b` (fast) or `openai/gpt-oss-120b` (stronger). Check live models at console.groq.com/docs/models.

## 1. invoke() — ask one question

In [ ]:
response = llm.invoke("Give me a short example of AI use in daily life?")

In [ ]:
print(response)

In [ ]:
print(response.content)

## 2. What is inside the response?

In [ ]:
type(response)

In [ ]:
response.response_metadata

In [ ]:
response.usage_metadata

In [ ]:
print("Input tokens :", response.usage_metadata["input_tokens"])
print("Output tokens:", response.usage_metadata["output_tokens"])
print("Total tokens :", response.usage_metadata["total_tokens"])

## 3. Try a bigger model

In [ ]:
llm_big = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
response = llm_big.invoke("Explain LangChain in 2 lines for a beginner")
print(response.content)

## 4. Model parameters
- **temperature** → creativity (0 = same answer every time, 1 = creative)
- **max_tokens** → maximum length of the answer
- **timeout / max_retries** → safety for production

In [ ]:
llm_factual = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
llm_creative = ChatGroq(model="openai/gpt-oss-20b", temperature=1)

In [ ]:
for i in range(2):
    print("factual :", llm_factual.invoke("Give a 4-word tagline for a coffee shop").content)

In [ ]:
for i in range(2):
    print("creative:", llm_creative.invoke("Give a 4-word tagline for a coffee shop").content)

In [ ]:
llm_short = ChatGroq(model="openai/gpt-oss-20b", max_tokens=300, timeout=30, max_retries=2)
print(llm_short.invoke("What is machine learning?").content)

## 5. Messages — give the model a role
| Message | Role | Use |
|---|---|---|
| `SystemMessage` | system | how the model should behave |
| `HumanMessage` | user | the question |
| `AIMessage` | assistant | the model's answer (or an example answer) |

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

In [ ]:
messages = [
    SystemMessage("You are a funny teacher. Answer in 2 lines."),
    HumanMessage("What is gravity?")
]

In [ ]:
response = llm.invoke(messages)
print(response.content)

In [ ]:
# same thing using simple dictionaries
messages = [
    {"role": "system", "content": "Reply only in Hindi."},
    {"role": "user", "content": "What is Python?"}
]
print(llm.invoke(messages).content)

### Few-shot with AIMessage (show examples to the model)

In [ ]:
messages = [
    SystemMessage("Classify the review as POSITIVE or NEGATIVE. One word only."),
    HumanMessage("The food was amazing"), AIMessage("POSITIVE"),
    HumanMessage("Delivery was very late"), AIMessage("NEGATIVE"),
    HumanMessage("Packaging was good but taste was bad"),
]
print(llm.invoke(messages).content)

## 6. stream() — print the answer word by word (like ChatGPT)

In [ ]:
for chunk in llm.stream("Write a 4-line poem on Mumbai rains"):
    print(chunk.content, end="", flush=True)

## 7. batch() — many questions at once (parallel)

In [ ]:
questions = ["What is AI?", "What is ML?", "What is Deep Learning?"]

In [ ]:
responses = llm.batch(questions)

In [ ]:
for q, r in zip(questions, responses):
    print(q, "->", r.content[:120], "\n")

## 8. ainvoke() — async (used in FastAPI / web apps)

In [ ]:
response = await llm.ainvoke("Say hello in Marathi")
print(response.content)

## 9. Model profile — what can this model do?
`llm.profile` tells the context window, tool calling support, structured output support etc. (data from models.dev, beta feature)

In [ ]:
llm.profile

In [ ]:
profile = llm.profile or {}
print("Context window :", profile.get("max_input_tokens"))
print("Max output     :", profile.get("max_output_tokens"))
print("Tool calling   :", profile.get("tool_calling"))
print("Structured out :", profile.get("structured_output"))
print("Image input    :", profile.get("image_inputs"))

## 10. init_chat_model — one function for every provider
Format: `"provider:model_name"`

In [ ]:
from langchain.chat_models import init_chat_model

In [ ]:
llm2 = init_chat_model("groq:openai/gpt-oss-20b", temperature=0)
print(llm2.invoke("Capital of Maharashtra?").content)

### Other providers (need their own key + package)
```python
# Gemini  →  ! pip install langchain-google-genai   ;  os.environ['GOOGLE_API_KEY'] = userdata.get('gemini_key')
llm = init_chat_model("google_genai:gemini-2.5-flash")

# OpenAI  →  ! pip install langchain-openai   ;  os.environ['OPENAI_API_KEY'] = userdata.get('openai_key')
llm = init_chat_model("openai:gpt-4o-mini")

# Ollama (free, runs on your laptop)  →  ! pip install langchain-ollama
llm = init_chat_model("ollama:llama3.1")
```

## 11. Make it modular — compare models with one function

In [ ]:
def test_model(model_name, query):
    model = ChatGroq(model=model_name, temperature=0)
    response = model.invoke(query)
    print(f"\n--- {model_name} ---")
    print(response.content[:300])
    print("tokens:", response.usage_metadata["total_tokens"])

for m in ["openai/gpt-oss-20b", "openai/gpt-oss-120b"]:
    test_model(m, "Explain RAG in one line")

# Benefits of LangChain Chat Models
- **Same code for all providers** → switch Groq ↔ OpenAI ↔ Gemini in one line
- **Built-in streaming, batching, async**
- **Standard response** → `.content`, `.usage_metadata` work everywhere
- **Plugs into everything else** → prompts, chains, agents, RAG

# Disadvantages / Care points
- Extra library layer over the raw API
- Some provider-only features need provider-specific parameters
- Model names change often → check provider docs

## 🧪 Practice
1. Ask the same question to `gpt-oss-20b` and `gpt-oss-120b` and compare tokens.
2. Create a `SystemMessage` "You are a strict interviewer" and ask a Python question.
3. Use `batch()` to translate 5 English sentences to Hindi.
4. Stream a 100-word story and count the chunks.